# M3 · Keypoints de ruedas → número de ejes
**IAM-811 · Laboratorio 3**

El modelo ubica un *keypoint* en el centro de cada rueda visible desde un lado del vehículo.
Número de ruedas visibles de un lado = número de ejes (las ruedas dobles comparten el centro).

**Dataset** (Roboflow, proyecto de tipo *Keypoint Detection*):
1. Crear el esqueleto con **1 punto**: `centro`. Clase: `rueda`.
2. En cada foto lateral (carpetas `autos_de_costado`, `camion`, `bus`, `moto`) dibujar una caja por rueda y poner el punto en el centro del aro.
3. Versión: Resize 640, aumentaciones: brillo ±20 %, desenfoque 1 px, rotación ±5° (x3). **No** usar flip vertical.
4. Exportar en formato **YOLOv8** (pose; compatible con YOLO11).

In [ ]:
# Versiones fijadas (mismas que inferencia/requirements.txt)
!pip install -q ultralytics==8.3.240 roboflow==1.1.66

In [ ]:
# Diagnóstico del entorno (punto adicional: gestión de entornos). Guardar la salida en la carpeta de evidencias.
import sys, platform, subprocess
import torch, ultralytics
print("python:", sys.version.split()[0])
print("sistema:", platform.platform())
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| gpu disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("ultralytics:", ultralytics.__version__)
subprocess.run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv", shell=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/peaje_evidencias'
import os; os.makedirs(SALIDA, exist_ok=True)

In [ ]:
# La API key se lee de los "Secretos" de Colab (icono de llave a la izquierda): nombre ROBOFLOW_API_KEY.
# NUNCA la escribas en el notebook ni la subas al repositorio.
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))

WORKSPACE = 'tu-workspace'          # <-- cambiar
PROYECTO = 'peaje-ruedas-keypoints'  # <-- cambiar
VERSION = 1                          # <-- cambiar
dataset = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov8')
print(open(f'{dataset.location}/data.yaml').read())

In [ ]:
# Roboflow exporta kpt_shape [1, 3] (x, y, visibilidad). flip_idx con un solo punto = [0].
import yaml
ruta = f'{dataset.location}/data.yaml'
cfg = yaml.safe_load(open(ruta))
cfg['kpt_shape'] = [1, 3]
cfg['flip_idx'] = [0]
yaml.safe_dump(cfg, open(ruta, 'w'))
import glob
for parte in ['train', 'valid', 'test']:
    n = len(glob.glob(f'{dataset.location}/{parte}/images/*'))
    r = sum(len(open(f).read().strip().splitlines()) for f in glob.glob(f'{dataset.location}/{parte}/labels/*.txt'))
    print(f'{parte}: {n} imágenes, {r} ruedas')

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11n-pose.pt')
modelo.train(data=f'{dataset.location}/data.yaml', epochs=100, imgsz=640, batch=16, patience=25,
             fliplr=0.5, project='/content/runs', name='m3_pose', seed=42)

In [ ]:
# Métricas: mAP de keypoints (pose) y de cajas.
mejor = YOLO('/content/runs/m3_pose/weights/best.pt')
m = mejor.val(data=f'{dataset.location}/data.yaml', split='val')
print(f'pose mAP50={m.pose.map50:.3f} mAP50-95={m.pose.map:.3f} | caja mAP50={m.box.map50:.3f}')

In [ ]:
# Error en el conteo de ejes: ejes reales (etiquetas) vs ejes predichos, con la misma regla del endpoint
# (ruedas a menos de 25 px en x se cuentan como un solo eje).
import glob, os, numpy as np, cv2

def ejes_desde_x(xs, umbral=25):
    xs = sorted(xs); ejes = []
    for x in xs:
        if not ejes or x - ejes[-1] > umbral:
            ejes.append(x)
    return len(ejes)

parte = 'test' if os.path.isdir(f'{dataset.location}/test/images') else 'valid'
errores, exactos, total = [], 0, 0
for img in glob.glob(f'{dataset.location}/{parte}/images/*'):
    base = os.path.splitext(os.path.basename(img))[0]
    lbl = f'{dataset.location}/{parte}/labels/{base}.txt'
    if not os.path.exists(lbl):
        continue
    h, w = cv2.imread(img).shape[:2]
    xs_reales = [float(l.split()[5]) * w for l in open(lbl) if l.strip()]
    reales = ejes_desde_x(xs_reales)
    r = mejor.predict(img, conf=0.35, verbose=False)[0]
    xs_pred = [kp[0][0] for kp in r.keypoints.xy.tolist()] if r.keypoints is not None else []
    pred = ejes_desde_x(xs_pred)
    errores.append(abs(pred - reales)); exactos += int(pred == reales); total += 1
print(f'Imágenes: {total} | conteo exacto: {exactos/total:.1%} | MAE ejes: {np.mean(errores):.2f}')

In [ ]:
import shutil
shutil.copy('/content/runs/m3_pose/weights/best.pt', f'{SALIDA}/m3_pose.pt')
shutil.copytree('/content/runs/m3_pose', f'{SALIDA}/m3_pose_run', dirs_exist_ok=True)
print(os.listdir(SALIDA))